In [6]:
import os
import shutil
from pathlib import Path
import albumentations as A
import cv2


In [4]:
DATA_DIR = Path("../data/splits")
OUTPUT_DIR = Path("../data/augmented")
LABELS = ["glioma", "meningioma", "notumor", "pituitary"]

In [7]:
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.05,
        rotate_limit=10,
        p=0.5
    ),
    A.RandomBrightnessContrast(
        brightness_limit=0.1,
        contrast_limit=0.1,
        p=0.5
    ),
    A.CLAHE(clip_limit=2.0, p=0.3)
])

c:\Users\noman\anaconda3\envs\mlai\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [9]:
test_valid_transform = A.Compose([
    A.Resize(224, 224)
])

In [10]:
for aug_split in ["train", "val", "test"]:
    for label in LABELS:
        input_dir = DATA_DIR/aug_split/label
        output_dir = OUTPUT_DIR/aug_split/label
        output_dir.mkdir(parents=True, exist_ok=True)
        for img_name in os.listdir(input_dir):
            if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
                img_path = input_dir/img_name
                img = cv2.imread(str(img_path))
                if img is None:
                        continue
                image = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                if aug_split == "train":
                    base_aug = A.Resize(224, 224)(image=image)["image"]
                    cv2.imwrite(
                        os.path.join(output_dir,f"resized_{img_name}"),
                        cv2.cvtColor(base_aug, cv2.COLOR_RGB2BGR)
                    )
                    for i in range(3):
                        augmented = train_transform(image=image)["image"]
                        
                        save_path = os.path.join(output_dir, f"aug_{i}_{img_name}")
                        cv2.imwrite(save_path, cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))
            
                elif aug_split in ["val", "test"]:
                    augmented = test_valid_transform(image=image)["image"]
                    cv2.imwrite(
                        os.path.join(output_dir,f"preprocessed_{img_name}"),
                        cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
                    )